# worker-batched スループットベンチマーク

`workers`と1 workerあたりのlane数をコード内で指定し、全構成共通の総試合数で順番に計測します。実際の`lanes`は `workers × lanes_per_worker` で自動計算します。

`IPC/NN評価 = IPC request数 / NN評価数` です。値が小さいほど、1回のプロセス間通信に多くのNN評価をまとめられています。例えば `0.125` は、平均すると1 IPCあたり8評価です。

現在の`worker-batched`実装では、queueに到着済みの要求だけを固定待機なしで回収し、追加要求を待たずにGPU評価します。workerは疎特徴量をNumPy連続配列にまとめ、中央batcherは`torch.from_numpy()`で共有してからCUDAまたはMPSへ転送します。worker jobは中央batchへ詰めやすいようMPSでは`batch_size / ceil(workers / 2)`、CUDAでは`batch_size / workers`を上限とし、中央で同じモデルのjobを`batch_size`まで結合します。複数モデルの疎埋め込みは1回の標準EmbeddingBagへ統合します。CUDAは平均model batch 32未満で従来の`torch.vmap` model-axis経路を使います。MPSは通常モデルのforwardをそのまま使う別経路で、同じwaveの全モデル入力をindex/offsetとvalueのdtype別にまとめ、モデル数にかかわらず2回のdevice転送へ削減します。value/policy出力もモデルごとに結合して1回でCPUへ戻し、全forwardを先に投入してモデルごとの同期を避けます。

CPU使用率とシステムRAMは`psutil`で採取します。GPU使用率はNVIDIAでは`nvidia-smi`、Apple SiliconのMPSでは`powermetrics`を約1秒間隔で採取します。MPSでは区間平均の`GPU HW active residency`を`GPU平均%`へ、中央runnerの`gputime_ms_per_s`を`runner GPU平均%`へ表示します。`powermetrics`には管理者権限が必要なため、実行セルの開始時にsudoパスワード入力欄が一度表示されます。パスワードはファイルやNotebook出力へ保存しません。生のplistは`results/`へ保存します。ベンチ親processと全子workerのRSS合計、空きRAM最小値も表へ表示します。中央NNのmerge、from_numpy/H2D、forward、完了待ち/D2Hと、worker側NumPy梱包の累積時間も表示します。`Search.step`はC API+ctypes、JSON decode+軽量object生成、従来dataclass変換へ分割して表示します。`msgspec`が入っていればJSONから軽量objectへ直接decodeするため、dataclass時間は0になります。MCTS nodeはdecode後のObservation全体を保持せず、`searchId`・手番・勝敗だけを保持します。複数選択の候補手では同じoptionのdecoder特徴を局面内で一度だけ生成して再利用します。

CUDA/MPS演算は非同期です。`forward秒`はCPUが`model(*inputs)`を呼んで戻るまで、`待ち+D2H秒`は直後の`.cpu()`でforward完了を同期して出力をGPUからCPUへコピーするまでの累積です。未完了のforward計算が後者へ計上されるため、純粋なGPU演算時間を見るときは2列を合計して比較します。

In [30]:
from __future__ import annotations

import getpass
import os
from pathlib import Path
import plistlib
import re
import signal
import subprocess
import sys
import time

import pandas as pd
import psutil
from IPython.display import display


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'tools' / 'run_matches_round_robin.py').exists():
            return candidate
    raise FileNotFoundError('pokemon-tcg-agent のリポジトリルートが見つかりません。')


ROOT = find_repo_root()
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

venv_python = ROOT / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
PYTHON = venv_python if venv_python.exists() else Path(sys.executable)
RUNNER = ROOT / 'tools' / 'run_matches_round_robin.py'

print(f'ROOT={ROOT}')
print(f'PYTHON={PYTHON}')

ROOT=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent
PYTHON=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/.venv/bin/python


## ベンチマーク条件

`BENCHMARKS`を編集して任意の`workers`と`lanes_per_worker`を指定します。実際の`lanes`は `workers × lanes_per_worker` です。試合数は全構成共通の`TOTAL_GAMES`で指定します。現在の学習ループと同様に自己対戦を含めて全組み合わせへ差1以内で均等配分し、`MAX_TURNS`または`MAX_SELECTIONS`へ達した試合は残りサイド枚数で勝敗を確定します。同数は引き分けです。構成比較では必ず同じ`TOTAL_GAMES`を使ってください。このNotebookは対戦性能だけを測り、学習JSONは保存しません。

In [31]:
AGENTS = {
    'cluster_00': 'agents/16model_result/cluster_00/src/main.py',
    'cluster_01': 'agents/16model_result/cluster_01/src/main.py',
    'cluster_02': 'agents/16model_result/cluster_02/src/main.py',
    'cluster_03': 'agents/16model_result/cluster_03/src/main.py',
    'cluster_04': 'agents/16model_result/cluster_04/src/main.py',
    'cluster_05': 'agents/16model_result/cluster_05/src/main.py',
    'cluster_06': 'agents/16model_result/cluster_06/src/main.py',
    'cluster_07': 'agents/16model_result/cluster_07/src/main.py',
    'cluster_08': 'agents/16model_result/cluster_08/src/main.py',
    'cluster_09': 'agents/16model_result/cluster_09/src/main.py',
    'cluster_10': 'agents/16model_result/cluster_10/src/main.py',
    'cluster_11': 'agents/16model_result/cluster_11/src/main.py',
    'cluster_12': 'agents/16model_result/cluster_12/src/main.py',
    'cluster_13': 'agents/16model_result/cluster_13/src/main.py',
    'cluster_14': 'agents/16model_result/cluster_14/src/main.py',
    'cluster_15': 'agents/16model_result/cluster_15/src/main.py',
}


# 論理CPUのうち1つを中央処理に残し、残りをworkerへ割り当てる。
CPU_THREADS = psutil.cpu_count(logical=True) or os.cpu_count() or 1
WORKERS = max(1, CPU_THREADS - 1)
LANES_PER_WORKER = 400

# 元のベンチマークと同様、全workerのlaneを一度に埋める。
TOTAL_GAMES = WORKERS * LANES_PER_WORKER

# 比較するworkersと1 workerあたりのlane数を指定する。
# 実際のlanesは workers * lanes_per_worker。
BENCHMARKS = [
    {'workers': WORKERS, 'lanes_per_worker': LANES_PER_WORKER},
]

BATCH_SIZE = 256
SEARCH_COUNT = 10
MAX_TURNS = 100
MAX_SELECTIONS = 500
DEVICE = 'mps' if sys.platform == 'darwin' else 'cuda'
MODEL_AXIS = True
SEED = 0
PAIR_COUNT = len(AGENTS) * (len(AGENTS) + 1) // 2
print(
    f'エージェント数={len(AGENTS)}, 対戦カード数={PAIR_COUNT}, '
    f'論理CPU={CPU_THREADS}, 中央=1, workers={WORKERS}, device={DEVICE}, '
    f'総試合数={TOTAL_GAMES}, 自己対戦あり, '
    f'上限={MAX_TURNS}ターン/{MAX_SELECTIONS}選択'
)

エージェント数=16, 対戦カード数=136, 論理CPU=10, 中央=1, workers=9, device=mps, 総試合数=3600, 自己対戦あり, 上限=100ターン/500選択


In [32]:
NN_PATTERN = re.compile(
    r'NN: ([\d.]+)秒 / (\d+)評価 / (\d+)batch '
    r'\(平均batch=([\d.]+), 最大=(\d+)\)'
)
SEARCH_PATTERN = re.compile(
    r'libcg Search: begin=([\d.]+)秒, step=([\d.]+)秒/(\d+)回, '
    r'cleanup=([\d.]+)秒'
)
BATTLE_PATTERN = re.compile(
    r'libcg Battle: start=([\d.]+)秒, step=([\d.]+)秒/(\d+)回, '
    r'特徴量生成=([\d.]+)秒'
)
SEARCH_STEP_PATTERN = re.compile(
    r'Search\.step内訳: C API\+ctypes=([\d.]+)秒, '
    r'JSON decode\+object=([\d.]+)秒, dataclass=([\d.]+)秒'
)
CENTRAL_NN_PATTERN = re.compile(
    r'中央NN内訳: merge/pad=([\d.]+)秒, '
    r'from_numpy/H2D=([\d.]+)秒, forward投入=([\d.]+)秒, '
    r'forward待ち\+D2H=([\d.]+)秒, tolist=([\d.]+)秒, '
    r'応答分割=([\d.]+)秒, response put=([\d.]+)秒'
)
IPC_PATTERN = re.compile(
    r'CPU workers: (\d+), worker NN待ち合計=([\d.]+)秒, '
    r'worker NumPy梱包=([\d.]+)秒, '
    r'中央batch収集=([\d.]+)秒, IPC request=(\d+)回'
)
CUDA_WAVE_PATTERN = re.compile(
    r'(?:Model|CUDA) wave: model-axis=(\d+), per-model=(\d+)'
)
DECODER_PADDING_PATTERN = re.compile(
    r'Decoder padding: source=(\d+) token, padded=(\d+) token, '
    r'有効率=([\d.]+)%'
)


def require_match(pattern: re.Pattern[str], output: str, label: str) -> re.Match[str]:
    match = pattern.search(output)
    if match is None:
        raise RuntimeError(f'{label}を実行結果から取得できません。')
    return match


def start_gpu_sampler() -> subprocess.Popen[str] | None:
    try:
        return subprocess.Popen(
            [
                'nvidia-smi',
                '--query-gpu=utilization.gpu',
                '--format=csv,noheader,nounits',
                '--loop-ms=1000',
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
            creationflags=subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0,
        )
    except (FileNotFoundError, OSError):
        return None


def stop_gpu_sampler(sampler: subprocess.Popen[str] | None) -> list[float]:
    if sampler is None:
        return []
    sampler.terminate()
    try:
        stdout, _ = sampler.communicate(timeout=5)
    except subprocess.TimeoutExpired:
        sampler.kill()
        stdout, _ = sampler.communicate()
    samples: list[float] = []
    for line in stdout.splitlines():
        try:
            samples.append(float(line.strip()))
        except ValueError:
            pass
    return samples


def average_or_nan(values: list[float]) -> float:
    return sum(values) / len(values) if values else float('nan')


def authorize_powermetrics() -> None:
    if sys.platform != 'darwin':
        return
    cached = subprocess.run(
        ['sudo', '-n', 'true'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
    if cached.returncode == 0:
        return
    password = getpass.getpass('powermetrics用のsudoパスワード: ')
    try:
        authenticated = subprocess.run(
            ['sudo', '-S', '-p', '', '-v'],
            input=password + '\n',
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
        )
    finally:
        password = ''
    if authenticated.returncode != 0:
        raise RuntimeError('sudo認証に失敗したためpowermetricsを開始できません。')


def start_mps_gpu_sampler(output_path: Path) -> subprocess.Popen[str]:
    authorize_powermetrics()
    # root所有のファイルを作らないよう、出力先は先に通常userで作成する。
    output_path.touch(exist_ok=False)
    return subprocess.Popen(
        [
            'sudo', '-n', '/usr/bin/powermetrics',
            '--samplers', 'gpu_power,tasks',
            '--show-process-gpu',
            '--sample-rate', '1000',
            '--buffer-size', '1',
            '--format', 'plist',
            '--output-file', str(output_path),
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
        start_new_session=True,
    )


def walk_mappings(value: object):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_mappings(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_mappings(child)


def parse_powermetrics(
    output_path: Path,
    runner_pid: int,
) -> tuple[list[float], list[float]]:
    records = []
    for payload in output_path.read_bytes().split(b'\0'):
        if not payload.strip():
            continue
        try:
            records.append(plistlib.loads(payload))
        except Exception:
            # 停止時に末尾のplistが書きかけなら、完成済みsampleだけを使う。
            continue

    gpu_active_samples: list[float] = []
    runner_gpu_samples: list[float] = []
    for record in records:
        gpu = record.get('gpu') if isinstance(record, dict) else None
        if isinstance(gpu, dict) and 'idle_ratio' in gpu:
            active_percent = 100.0 * (1.0 - float(gpu['idle_ratio']))
            gpu_active_samples.append(min(100.0, max(0.0, active_percent)))
        task_records = record.get('tasks', []) if isinstance(record, dict) else []
        for mapping in walk_mappings(task_records):
            process_id = mapping.get('pid')
            if process_id != runner_pid or 'gputime_ms_per_s' not in mapping:
                continue
            # 1000 GPU ms/sを100%として割合へ変換する。
            runner_gpu_samples.append(
                min(100.0, max(0.0, float(mapping['gputime_ms_per_s']) / 10.0))
            )
            break
    return gpu_active_samples, runner_gpu_samples


def stop_mps_gpu_sampler(
    sampler: subprocess.Popen[str],
    output_path: Path,
    runner_pid: int,
) -> tuple[list[float], list[float]]:
    try:
        os.killpg(sampler.pid, signal.SIGINT)
    except ProcessLookupError:
        pass
    try:
        _, stderr = sampler.communicate(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(sampler.pid, signal.SIGKILL)
        _, stderr = sampler.communicate()
    if not output_path.exists():
        raise RuntimeError(f'powermetricsの出力がありません: {stderr[-1000:]}')
    gpu_samples, runner_samples = parse_powermetrics(output_path, runner_pid)
    if not gpu_samples:
        raise RuntimeError(f'GPU active residencyを取得できません: {stderr[-1000:]}')
    return gpu_samples, runner_samples


def process_tree_rss_gib(process_id: int) -> float | None:
    try:
        root = psutil.Process(process_id)
        processes = [root, *root.children(recursive=True)]
        rss = 0
        for child in processes:
            try:
                rss += child.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        return rss / 1024**3
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return None


def run_benchmark(config: dict[str, int]) -> dict[str, float | int | str]:
    workers = int(config['workers'])
    lanes_per_worker = int(config['lanes_per_worker'])
    lanes = workers * lanes_per_worker
    total_games = TOTAL_GAMES
    command = [str(PYTHON), str(RUNNER)]
    for name, agent_path in AGENTS.items():
        command.extend(['--agent', f'{name}={agent_path}'])
    command.extend([
        '--backend', 'worker-batched',
        '--device', DEVICE,
        '--workers', str(workers),
        '--lanes', str(lanes),
        '--total-games', str(total_games),
        '--batch-size', str(BATCH_SIZE),
        '--search-count', str(SEARCH_COUNT),
        '--max-turns', str(MAX_TURNS),
        '--max-selections', str(MAX_SELECTIONS),
        '--seed', str(SEED),
        '--quiet',
    ])
    if not MODEL_AXIS:
        command.append('--no-model-axis')
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    cpu_samples: list[float] = []
    ram_samples: list[float] = []
    available_ram_samples: list[float] = []
    process_rss_samples: list[float] = []
    gpu_samples: list[float] = []
    runner_gpu_samples: list[float] = []
    psutil.cpu_percent(interval=None)  # 最初の差分計測を初期化
    gpu_sampler = None if DEVICE == 'mps' else start_gpu_sampler()
    if DEVICE == 'mps':
        authorize_powermetrics()
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    log_prefix = f'notebook_w{workers}_l{lanes}_g{total_games}_{timestamp}'
    powermetrics_path = RESULTS_DIR / f'{log_prefix}.powermetrics.plist'
    mps_gpu_sampler = (
        start_mps_gpu_sampler(powermetrics_path)
        if DEVICE == 'mps'
        else None
    )
    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding='utf-8',
        errors='replace',
        creationflags=subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0,
    )
    try:
        while process.poll() is None:
            cpu_samples.append(psutil.cpu_percent(interval=1.0))
            memory = psutil.virtual_memory()
            ram_samples.append(memory.percent)
            available_ram_samples.append(memory.available / 1024**3)
            process_rss = process_tree_rss_gib(process.pid)
            if process_rss is not None:
                process_rss_samples.append(process_rss)
        stdout, stderr = process.communicate()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.communicate(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.communicate()
        raise
    finally:
        wall = time.perf_counter() - started
        if mps_gpu_sampler is not None:
            gpu_samples, runner_gpu_samples = stop_mps_gpu_sampler(
                mps_gpu_sampler,
                powermetrics_path,
                process.pid,
            )
        else:
            gpu_samples.extend(stop_gpu_sampler(gpu_sampler))

    (RESULTS_DIR / f'{log_prefix}.stdout.log').write_text(
        stdout, encoding='utf-8'
    )
    (RESULTS_DIR / f'{log_prefix}.stderr.log').write_text(
        stderr, encoding='utf-8'
    )
    if process.returncode != 0:
        raise RuntimeError(
            f'w={workers}, lanes={lanes} がexit={process.returncode}で失敗しました。\n'
            f'{stderr[-2000:]}'
        )

    nn = require_match(NN_PATTERN, stdout, 'NN profile')
    search = require_match(SEARCH_PATTERN, stdout, 'Search profile')
    battle = require_match(BATTLE_PATTERN, stdout, 'Battle profile')
    search_step = require_match(SEARCH_STEP_PATTERN, stdout, 'Search.step profile')
    central_nn = require_match(CENTRAL_NN_PATTERN, stdout, 'central NN profile')
    ipc = require_match(IPC_PATTERN, stdout, 'IPC profile')
    cuda_wave = require_match(CUDA_WAVE_PATTERN, stdout, 'CUDA wave profile')
    decoder_padding = require_match(
        DECODER_PADDING_PATTERN, stdout, 'decoder padding profile'
    )

    nn_evaluations = int(nn.group(2))
    battle_steps = int(battle.group(3))
    ipc_requests = int(ipc.group(5))
    return {
        'workers': workers,
        'lanes': lanes,
        'games': total_games,
        'pairings': PAIR_COUNT,
        'self': 'yes',
        'max turns': MAX_TURNS,
        'max selections': MAX_SELECTIONS,
        'wall': wall,
        'games/s': total_games / wall,
        'Battle step/s': battle_steps / wall,
        '平均batch': float(nn.group(4)),
        'IPC/NN評価': ipc_requests / nn_evaluations,
        'NN秒': float(nn.group(1)),
        'Search begin秒': float(search.group(1)),
        'Search step秒': float(search.group(2)),
        'Search cleanup秒': float(search.group(4)),
        'Battle step秒': float(battle.group(2)),
        '特徴量生成秒': float(battle.group(4)),
        'merge/pad秒': float(central_nn.group(1)),
        'from_numpy/H2D秒': float(central_nn.group(2)),
        'forward秒': float(central_nn.group(3)),
        '待ち+D2H秒': float(central_nn.group(4)),
        'worker NumPy梱包秒': float(ipc.group(3)),
        'Search C API秒': float(search_step.group(1)),
        'Search JSON/object秒': float(search_step.group(2)),
        'Search dataclass秒': float(search_step.group(3)),
        'model-axis wave': int(cuda_wave.group(1)),
        'per-model wave': int(cuda_wave.group(2)),
        'model-axis比率': int(cuda_wave.group(1)) / max(
            int(cuda_wave.group(1)) + int(cuda_wave.group(2)), 1
        ),
        'decoder token有効率': float(decoder_padding.group(3)) / 100,
        'CPU平均%': average_or_nan(cpu_samples),
        'CPU最大%': max(cpu_samples, default=float('nan')),
        'RAM平均%': average_or_nan(ram_samples),
        'RAM最大%': max(ram_samples, default=float('nan')),
        '空きRAM最小GiB': min(available_ram_samples, default=float('nan')),
        'process RSS平均GiB': average_or_nan(process_rss_samples),
        'process RSS最大GiB': max(process_rss_samples, default=float('nan')),
        'GPU平均%': average_or_nan(gpu_samples),
        'GPU最大%': max(gpu_samples, default=float('nan')),
        'runner GPU平均%': average_or_nan(runner_gpu_samples),
        'runner GPU最大%': max(runner_gpu_samples, default=float('nan')),
        'GPU計測': 'powermetrics' if DEVICE == 'mps' else 'nvidia-smi',
        'GPU計測ファイル': str(powermetrics_path) if DEVICE == 'mps' else '',
    }

## 実行

構成を直列に実行します。複数構成を同時に動かすと同じGPUを奪い合うため、ベンチマークとして比較できなくなります。

In [33]:
blocking_processes = []
for candidate in psutil.process_iter(['pid', 'cmdline']):
    if candidate.info['pid'] == os.getpid():
        continue
    command_text = ' '.join(candidate.info.get('cmdline') or [])
    if (
        'run_matches_round_robin.py' in command_text
        or 'train_parallel_generation.py' in command_text
    ):
        blocking_processes.append((candidate.info['pid'], command_text))
if blocking_processes:
    raise RuntimeError(
        '別の対戦・学習が動いているため、終了後に測定してください: '
        + ', '.join(str(pid) for pid, _ in blocking_processes)
    )

rows = []
for index, config in enumerate(BENCHMARKS, start=1):
    print(
        f'[{index}/{len(BENCHMARKS)}] '
        f"workers={config['workers']}, "
        f"lanes_per_worker={config['lanes_per_worker']}, "
        f"lanes={config['workers'] * config['lanes_per_worker']}, "
        f"games={TOTAL_GAMES}"
    )
    row = run_benchmark(config)
    rows.append(row)
    print(
        f"  wall={row['wall']:.1f}s, games/s={row['games/s']:.3f}, "
        f"Battle step/s={row['Battle step/s']:.1f}, "
        f"平均batch={row['平均batch']:.1f}, IPC/NN評価={row['IPC/NN評価']:.4f}, "
        f"CPU平均={row['CPU平均%']:.1f}%, GPU平均={row['GPU平均%']:.1f}%, "
        f"runner GPU平均={row['runner GPU平均%']:.1f}%, "
        f"RAM最大={row['RAM最大%']:.1f}%, "
        f"process RSS最大={row['process RSS最大GiB']:.2f}GiB"
    )

results = pd.DataFrame(rows).sort_values('games/s', ascending=False).reset_index(drop=True)

[1/1] workers=9, lanes_per_worker=400, lanes=3600, games=3600
  wall=357.4s, games/s=10.073, Battle step/s=1667.6, 平均batch=27.5, IPC/NN評価=0.0067, CPU平均=62.1%, GPU平均=0.9%, RAM最大=86.1%, process RSS最大=5.02GiB


In [34]:
display(
    results.style
    .format({
        'wall': '{:.3f}',
        'games/s': '{:.3f}',
        'Battle step/s': '{:.1f}',
        '平均batch': '{:.1f}',
        'IPC/NN評価': '{:.4f}',
        'NN秒': '{:.3f}',
        'Search begin秒': '{:.3f}',
        'Search step秒': '{:.3f}',
        'Search cleanup秒': '{:.3f}',
        'Battle step秒': '{:.3f}',
        '特徴量生成秒': '{:.3f}',
        'merge/pad秒': '{:.3f}',
        'from_numpy/H2D秒': '{:.3f}',
        'forward秒': '{:.3f}',
        '待ち+D2H秒': '{:.3f}',
        'worker NumPy梱包秒': '{:.3f}',
        'Search C API秒': '{:.3f}',
        'Search JSON/object秒': '{:.3f}',
        'Search dataclass秒': '{:.3f}',
        'model-axis比率': '{:.1%}',
        'decoder token有効率': '{:.1%}',
        'CPU平均%': '{:.1f}',
        'CPU最大%': '{:.1f}',
        'RAM平均%': '{:.1f}',
        'RAM最大%': '{:.1f}',
        '空きRAM最小GiB': '{:.2f}',
        'process RSS平均GiB': '{:.2f}',
        'process RSS最大GiB': '{:.2f}',
        'GPU平均%': '{:.1f}',
        'GPU最大%': '{:.1f}',
        'runner GPU平均%': '{:.1f}',
        'runner GPU最大%': '{:.1f}',
    })
    .background_gradient(subset=['games/s', 'Battle step/s', '平均batch'], cmap='YlGn')
    .background_gradient(
        subset=[
            'wall', 'IPC/NN評価', 'NN秒', 'Search begin秒',
            'Search step秒', 'Search cleanup秒', 'Battle step秒',
            '特徴量生成秒', 'merge/pad秒',
            'from_numpy/H2D秒', 'forward秒', '待ち+D2H秒',
            'worker NumPy梱包秒', 'Search C API秒',
            'Search JSON/object秒', 'Search dataclass秒',
        ],
        cmap='YlOrRd_r',
    )
    .background_gradient(
        subset=[
            'CPU平均%', 'GPU平均%', 'runner GPU平均%',
            'RAM平均%', 'RAM最大%',
            'process RSS平均GiB', 'process RSS最大GiB',
        ],
        cmap='Blues',
    )
)

,workers,lanes,games,pairings,self,max turns,max selections,wall,games/s,Battle step/s,平均batch,IPC/NN評価,NN秒,Search begin秒,Search step秒,Search cleanup秒,Battle step秒,特徴量生成秒,merge/pad秒,from_numpy/H2D秒,forward秒,待ち+D2H秒,worker NumPy梱包秒,Search C API秒,Search JSON/object秒,Search dataclass秒,model-axis wave,per-model wave,model-axis比率,decoder token有効率,CPU平均%,CPU最大%,RAM平均%,RAM最大%,空きRAM最小GiB,process RSS平均GiB,process RSS最大GiB,GPU平均%,GPU最大%
0,9,3600,3600,136,yes,100,500,357.398,10.073,1667.6,27.5,0.0067,263.839,69.911,142.740,0.036,92.524,222.877,6.535,11.607,159.854,84.785,65.720,83.161,59.579,0.000,0,11100,0.0%,69.4%,62.1,98.4,81.1,86.1,3.33,3.56,5.02,0.9,100.0
